In [ ]:
# Cell 1: Imports and Data Loading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Load the dataset directly from Kaggle's environment pathway
df = pd.read_csv('/kaggle/input/datasets/nalisha/tesla-ea-deliveries-and-production-data20152025/tesla_deliveries_dataset_2015_2025.csv')
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# Cell 2: Exploratory Data Analysis
print("--- Missing Values Check ---")
print(df.isnull().sum())

# Generate a correlation heatmap for structural checking
plt.figure(figsize=(10, 6))
numeric_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
# Create a continuous chronological timeline tracking metric
df['Timeline_Score'] = df['Year'] + (df['Month'] / 12.0)

# Sort the data chronologically
df = df.sort_values(by=['Year', 'Month']).reset_index(drop=True)

# Separate independent target configurations
X = df.drop(columns=['Estimated_Deliveries'])
y = df['Estimated_Deliveries']

# FIX: Automatically find which columns are text/object types and which are numeric
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Numerical features to scale:", numerical_cols)
print("Categorical features to encode:", categorical_cols)

In [ ]:
# 1. Processing rules for continuous measurements
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Processing rules for textual parameters (FIX: sparse_output=False)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Merging pathways into a universal preprocessor step
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

print("Universal preprocessing layout updated.")

In [ ]:
# Cell 5: Sequential Validation Matrix Setup
# Split linearly without data shuffling to simulate forecasting boundaries
split_idx = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training observations size: {X_train.shape[0]}")
print(f"Validation target testing size: {X_test.shape[0]}")

In [ ]:
# Instantiate modeling pipelines wrapping the updated preprocessor
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('regressor', RandomForestRegressor(random_state=42))])

# FIX: Added enable_categorical=True to ensure compatibility
xgb_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('regressor', XGBRegressor(random_state=42, enable_categorical=True))])

# Execute end-to-end fitting
print("Training Random Forest Pipeline...")
rf_pipeline.fit(X_train, y_train)

print("Training XGBoost Pipeline...")
xgb_pipeline.fit(X_train, y_train)
print("Both pipelines trained successfully!")

In [ ]:
# Cell 7: Automated Hyperparameter Optimization
# Setup a time series split pattern for validation folds
tscv = TimeSeriesSplit(n_splits=3)

# Establish search configurations targeted specifically at the XGBoost step
param_distributions = {
    'regressor__n_estimators': [50, 100, 200],
    'regressor__max_depth': [3, 5, 7],
    'regressor__learning_rate': [0.01, 0.1, 0.2]
}

# Run optimization using our time-series validation setup
tuned_xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=5,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1
)

print("Searching for optimized hyperparameters...")
tuned_xgb_search.fit(X_train, y_train)
best_model = tuned_xgb_search.best_estimator_
print("Optimal settings found:", tuned_xgb_search.best_params_)


In [ ]:
# Cell 8: Final Model Evaluation and Performance Plots
# Execute pipeline forecasting predictions
rf_preds = rf_pipeline.predict(X_test)
xgb_preds = best_model.predict(X_test)

# Calculate testing benchmarks
print(f"Random Forest - MAE: {mean_absolute_error(y_test, rf_preds):.2f} | R2 Score: {r2_score(y_test, rf_preds):.4f}")
print(f"Optimized XGBoost - MAE: {mean_absolute_error(y_test, xgb_preds):.2f} | R2 Score: {r2_score(y_test, xgb_preds):.4f}")

# Plot true performance curves against our forecasting results
plt.figure(figsize=(12, 6))
plt.plot(y_test.values, label="True Tesla Deliveries", color='black', marker='o')
plt.plot(xgb_preds, label="Optimized XGBoost Prediction", color='crimson', linestyle='--')
plt.title("Tesla Delivery Tracking: True Values vs Model Predictions")
plt.ylabel("Units")
plt.legend()
plt.grid(True)
plt.show()